# Kaggle Complete Notebook: EagleVision Phase 1 Adapted-Only Long-Run Training

This notebook runs an adapted-only Phase 1 workflow for EagleVision on Kaggle.

It does all of the following in one place:

1. clone and install the repository
2. inspect the Kaggle dataset
3. normalize the dataset into the ScanNet-style layout expected by the repo
4. download a Depth Anything V2 checkpoint
5. generate Kaggle-local train/eval configs (ablation-best settings)
6. train the Phase 1 adaptation model for a long run
7. sweep checkpoints to select the best converged model
8. run final adapted evaluation with the selected checkpoint
9. summarize adapted metrics and export artifacts
10. run adapted-checkpoint inference helpers for future cross-dataset comparisons

Confirmed dataset path for this notebook:

- `/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d`


## What Improvement We Are Testing

The proposed improvement is not a new learned renderer. The proposed improvement is a geometry-first training environment for improving a monocular depth estimator.

This notebook now focuses on:

- training a lightweight residual adapter on top of Depth Anything V2
- selecting a converged checkpoint via validation metric sweep
- preparing that trained adapted model for future inference comparisons across datasets and image quality settings

The key claim is that the adapted model should become more geometrically useful while preserving reasonable raw depth quality.


In [1]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import zipfile
import torch
import yaml


In [2]:
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

REPO_URL = 'https://github.com/alooboii/EagleVision.git'
REPO_DIR = KAGGLE_WORKING / 'EagleVision'

# Confirmed dataset path for this Kaggle runtime
RAW_DATASET_DIR = Path('/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d')

print('KAGGLE_INPUT:', KAGGLE_INPUT)
print('KAGGLE_WORKING:', KAGGLE_WORKING)
print('RAW_DATASET_DIR:', RAW_DATASET_DIR)

if not RAW_DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset path not found: {RAW_DATASET_DIR}')


KAGGLE_INPUT: /kaggle/input
KAGGLE_WORKING: /kaggle/working
RAW_DATASET_DIR: /kaggle/input/datasets/klein2111/scannet-2d/scannet_2d


## Shell Helper

The helper below is deliberately non-fragile for Kaggle. It can capture stdout and stderr without hiding the actual failure reason when a command exits nonzero.


In [3]:
def run(cmd, cwd=None, capture=False, check=True):
    print('$', ' '.join(cmd))
    print(f'[status] cwd={cwd if cwd else Path.cwd()}')
    print('[status] command started')

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    output_lines = []
    if process.stdout is not None:
        for line in process.stdout:
            print(line, end='')
            output_lines.append(line)

    return_code = process.wait()
    print(f'[status] command finished with exit code {return_code}')

    completed = subprocess.CompletedProcess(
        args=cmd,
        returncode=return_code,
        stdout=''.join(output_lines),
        stderr='',
    )

    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {' '.join(cmd)}"
        )
    return completed


def is_git_repo(path: Path) -> bool:
    return (path / '.git').exists()


## Clone Or Refresh The Repository

This cell is careful about Kaggle workspaces where a leftover directory may exist without actually being a git checkout.


In [4]:
if REPO_DIR.exists() and not is_git_repo(REPO_DIR):
    print(f'Removing non-git directory at {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

if is_git_repo(REPO_DIR):
    run(['git', 'fetch', '--all'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])

run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], cwd=REPO_DIR)
print('repo ready:', REPO_DIR)


$ git clone https://github.com/alooboii/EagleVision.git /kaggle/working/EagleVision
[status] cwd=/kaggle/working
[status] command started
Cloning into '/kaggle/working/EagleVision'...
[status] command finished with exit code 0
$ python -m pip install -q -e .[dev]
[status] cwd=/kaggle/working/EagleVision
[status] command started
[status] command finished with exit code 0
repo ready: /kaggle/working/EagleVision


In [5]:
TRAIN_RUN_OUTPUT_DIR = REPO_DIR / 'outputs' / 'phase1_kaggle_complete'
CONFIG_DIR = REPO_DIR / 'outputs' / 'kaggle_configs'
TARGET_DATA_ROOT = REPO_DIR / 'data' / 'scannet'

for path in [TRAIN_RUN_OUTPUT_DIR, CONFIG_DIR, TARGET_DATA_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# Make package imports robust in notebook kernels.
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ['PYTHONPATH'] = f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"

print('TRAIN_RUN_OUTPUT_DIR:', TRAIN_RUN_OUTPUT_DIR)
print('CONFIG_DIR:', CONFIG_DIR)
print('TARGET_DATA_ROOT:', TARGET_DATA_ROOT)
print('SRC_DIR:', SRC_DIR)


TRAIN_RUN_OUTPUT_DIR: /kaggle/working/EagleVision/outputs/phase1_kaggle_complete
CONFIG_DIR: /kaggle/working/EagleVision/outputs/kaggle_configs
TARGET_DATA_ROOT: /kaggle/working/EagleVision/data/scannet
SRC_DIR: /kaggle/working/EagleVision/src


## Inspect The Dataset

Before making any assumptions, inspect the first part of the directory tree. The screenshot you provided indicates scene folders with `color`, `depth`, `label`, and `pose`, which is exactly what we want.


In [6]:
preview = [str(path) for path in sorted(RAW_DATASET_DIR.iterdir())[:80]]
preview


['/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/intrinsics.txt',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0000_00',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0000_01',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0000_02',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0001_00',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0001_01',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0002_00',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0002_01',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0003_00',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0003_01',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0003_02',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0004_00',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene0005_00',
 '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d/scene

## Discover ScanNet-Style Scenes

We expect each scene to contain:

- `color/`
- `depth/`
- `pose/`

The extra `label/` directory is ignored.


In [7]:
def discover_scene_dirs(root: Path):
    scenes = []
    for candidate in sorted(root.iterdir()):
        if not candidate.is_dir():
            continue
        if (candidate / 'color').exists() and (candidate / 'depth').exists() and (candidate / 'pose').exists():
            scenes.append(candidate)
    return scenes

scene_dirs = discover_scene_dirs(RAW_DATASET_DIR)
print('num scenes found:', len(scene_dirs))
print('first scenes:', [p.name for p in scene_dirs[:10]])

if not scene_dirs:
    raise RuntimeError('No valid scene directories found.')


num scenes found: 1513
first scenes: ['scene0000_00', 'scene0000_01', 'scene0000_02', 'scene0001_00', 'scene0001_01', 'scene0002_00', 'scene0002_01', 'scene0003_00', 'scene0003_01', 'scene0003_02']


## Normalize Into `data/scannet`

The repository expects data to live under `data/scannet`. We symlink scenes where possible and copy only if necessary.


In [8]:
def materialize_scene(scene_dir: Path, target_root: Path):
    dst = target_root / scene_dir.name
    if dst.exists():
        return dst
    try:
        os.symlink(scene_dir, dst, target_is_directory=True)
    except OSError:
        shutil.copytree(scene_dir, dst)
    return dst

materialized = [materialize_scene(scene_dir, TARGET_DATA_ROOT) for scene_dir in scene_dirs]
scene_ids = sorted([p.name for p in TARGET_DATA_ROOT.iterdir() if p.is_dir()])

print('normalized scene count:', len(scene_ids))
print('example normalized scenes:', scene_ids[:10])


normalized scene count: 1513
example normalized scenes: ['scene0000_00', 'scene0000_01', 'scene0000_02', 'scene0001_00', 'scene0001_01', 'scene0002_00', 'scene0002_01', 'scene0003_00', 'scene0003_01', 'scene0003_02']


In [9]:
first_scene = TARGET_DATA_ROOT / scene_ids[0]
print('scene:', first_scene.name)
print('color sample:', [p.name for p in sorted((first_scene / 'color').glob('*'))[:5]])
print('depth sample:', [p.name for p in sorted((first_scene / 'depth').glob('*'))[:5]])
print('pose sample:', [p.name for p in sorted((first_scene / 'pose').glob('*'))[:5]])


scene: scene0000_00
color sample: ['0.jpg', '100.jpg', '1000.jpg', '1020.jpg', '1040.jpg']
depth sample: ['0.png', '100.png', '1000.png', '1020.png', '1040.png']
pose sample: ['0.txt', '100.txt', '1000.txt', '1020.txt', '1040.txt']


## Train / Validation Split

We create a simple scene-based split for the Kaggle run.


In [10]:
all_scene_ids = list(scene_ids)
print('total normalized scenes available:', len(all_scene_ids))
print('scene sample:', all_scene_ids[:10])


total normalized scenes available: 1513
scene sample: ['scene0000_00', 'scene0000_01', 'scene0000_02', 'scene0001_00', 'scene0001_01', 'scene0002_00', 'scene0002_01', 'scene0003_00', 'scene0003_01', 'scene0003_02']


## Download The Baseline Checkpoint

For Kaggle, we use the `vits` metric-depth model with the `hypersim` indoor profile.


In [11]:
print('[status] downloading Depth Anything V2 checkpoints for vits (metric + relative)')
run(
    [
        'python', '-m', 'baseline.depth_anything_v2', 'download',
        '--mode', 'all',
        '--profile', 'hypersim',
        '--encoder', 'vits',
    ],
    cwd=REPO_DIR,
)


[status] downloading Depth Anything V2 checkpoints for vits (metric + relative)
$ python -m baseline.depth_anything_v2 download --mode all --profile hypersim --encoder vits
[status] cwd=/kaggle/working/EagleVision
[status] command started
relative:vits -> /kaggle/working/EagleVision/baseline/depth_anything_v2/checkpoints/depth_anything_v2_vits.pth (downloaded)
metric:hypersim:vits -> /kaggle/working/EagleVision/baseline/depth_anything_v2/checkpoints/depth_anything_v2_metric_hypersim_vits.pth (downloaded)
[status] command finished with exit code 0


CompletedProcess(args=['python', '-m', 'baseline.depth_anything_v2', 'download', '--mode', 'all', '--profile', 'hypersim', '--encoder', 'vits'], returncode=0, stdout='relative:vits -> /kaggle/working/EagleVision/baseline/depth_anything_v2/checkpoints/depth_anything_v2_vits.pth (downloaded)\nmetric:hypersim:vits -> /kaggle/working/EagleVision/baseline/depth_anything_v2/checkpoints/depth_anything_v2_metric_hypersim_vits.pth (downloaded)\n', stderr='')

In [12]:
checkpoint_dir = REPO_DIR / 'baseline' / 'depth_anything_v2' / 'checkpoints'
checkpoint_files = list(checkpoint_dir.glob('*.pth'))
print('checkpoint dir:', checkpoint_dir)
print('checkpoint files:', checkpoint_files)

if not checkpoint_files:
    raise RuntimeError('No checkpoint file found after download.')


checkpoint dir: /kaggle/working/EagleVision/baseline/depth_anything_v2/checkpoints
checkpoint files: [PosixPath('/kaggle/working/EagleVision/baseline/depth_anything_v2/checkpoints/depth_anything_v2_vits.pth'), PosixPath('/kaggle/working/EagleVision/baseline/depth_anything_v2/checkpoints/depth_anything_v2_metric_hypersim_vits.pth')]


## Build Kaggle-Local Train And Eval Configs

Edit only the next cell (`Single Control Cell`).

This notebook is adapted-only: it does **not** run baseline evaluation. It trains, sweeps checkpoints for convergence, evaluates the selected adapted checkpoint, and prepares inference artifacts for future comparisons.


In [ ]:
# ===================== Single Control Cell (Edit Only Here) =====================
# Adapted-only workflow in this notebook:
# 1) Build train/eval configs with fixed pairing/model settings
# 2) Train longer with ablation-best hyperparameters
# 3) Sweep checkpoints to select best converged model
# 4) Run final adapted evaluation and save artifacts


EXPERIMENT_TAG = 'phase1_kaggle_best_longrun_v1'
SEED = 7

SCENE_LIMIT = None                 # use all discovered scenes
TRAIN_SCENE_FRACTION = 0.90
IMAGE_SIZE = [192, 288]            # if OOM, drop to [176, 256]

# Choose depth mode: 'metric' or 'relative'
DEPTH_MODE = 'metric'
DEPTH_ENCODER = 'vits'
METRIC_PROFILE = 'hypersim'  # used only when DEPTH_MODE == 'metric'

# Pair sampling controls (unchanged from validated settings)
PAIRING = {
    'min_translation_m': 0.02,
    'max_translation_m': 0.30,
    'min_rotation_deg': 0.8,
    'max_rotation_deg': 8.0,
    'max_index_gap': 10,
    'frame_stride': 2,
    'max_frames_per_scene': 180,
    'max_pairs_per_scene': 120,
}

device_name = "cuda" if torch.cuda.is_available() else "cpu"


# Resolve pretrained checkpoint according to selected mode.
if DEPTH_MODE == 'metric':
    ckpt_name = f'depth_anything_v2_metric_{METRIC_PROFILE}_{DEPTH_ENCODER}.pth'
else:
    ckpt_name = f'depth_anything_v2_{DEPTH_ENCODER}.pth'
DEFAULT_DEPTH_CKPT = REPO_DIR / 'baseline' / 'depth_anything_v2' / 'checkpoints' / ckpt_name
if not DEFAULT_DEPTH_CKPT.exists():
    raise FileNotFoundError(
        f'Missing pretrained DA-V2 checkpoint: {DEFAULT_DEPTH_CKPT}. Run the download cell first.'
    )

# Depth model controls (shared by train and eval)
DEPTH_MODEL_SETTINGS = {
    'mode': DEPTH_MODE,
    'encoder': DEPTH_ENCODER,
    'profile': METRIC_PROFILE,
    'checkpoint_path': str(DEFAULT_DEPTH_CKPT),
    'freeze_backbone': True,
    'adapter_hidden_channels': 32,
}


NO_EVAL_DURING_TRAIN = True

# Long-run training with ablation-best overall LR
TRAIN_SETTINGS = {
    'batch_size': 1,
    'epochs': 100,
    'max_steps_per_epoch': None,
    'lr': 5e-5,
    'weight_decay': 1e-4,
    'log_interval': 100,
    'vis_interval': 10_000_000,          # disable panel rendering overhead
    'checkpoint_interval': 10_000_000,   # keep epoch-end checkpoints only
}

# Final evaluation cap
EVAL_SETTINGS = {
    'batch_size': 1,
    'max_batches': 780,
}

# Convergence sweep controls
CHECKPOINT_SWEEP_ENABLED = True
SWEEP_PRIMARY_METRIC = 'abs_rel'          # lower is better
SWEEP_COARSE_MAX_BATCHES = 300
SWEEP_FULL_MAX_BATCHES = 780
SWEEP_EPOCH_STRIDE = 5
SWEEP_TOPK_FOR_FULL_EVAL = 5


# In relative mode, disable metric-depth losses by default (scale ambiguity).
if DEPTH_MODE == 'relative':
    LOSS_WEIGHTS = {
        'target_rgb': 1.0,
        'cycle_rgb': 1.0,
        'cycle_depth': 0.0,
        'target_depth': 0.0,
    }
else:
    LOSS_WEIGHTS = {
        'target_rgb': 1.0,
        'cycle_rgb': 1.0,
        'cycle_depth': 0.35,
        'target_depth': 0.35,
    }

# ScanNet default intrinsics used by this notebook's resized samples
intrinsics = [
    [577.8706, 0.0, 159.5],
    [0.0, 577.8706, 119.5],
    [0.0, 0.0, 1.0],
]

scene_pool = list(all_scene_ids if 'all_scene_ids' in globals() else scene_ids)
if len(scene_pool) < 2:
    raise RuntimeError('Need at least 2 scenes for train/val split.')

if SCENE_LIMIT is not None:
    scene_ids = scene_pool[:SCENE_LIMIT]
else:
    scene_ids = scene_pool

split_index = max(1, int(TRAIN_SCENE_FRACTION * len(scene_ids)))
train_scenes = scene_ids[:split_index]
val_scenes = scene_ids[split_index:] if split_index < len(scene_ids) else scene_ids[:1]

TRAIN_RUN_OUTPUT_DIR = REPO_DIR / 'outputs' / EXPERIMENT_TAG
TRAIN_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_config = {
    'seed': SEED,
    'device': device_name,
    'output_dir': str(TRAIN_RUN_OUTPUT_DIR),
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': IMAGE_SIZE,
        'intrinsics': intrinsics,
        'pairing': dict(PAIRING),
        'splits': {
            'train': {'scenes': train_scenes},
        },
    },
    'model': {
        'depth': dict(DEPTH_MODEL_SETTINGS),
    },
    'train': dict(TRAIN_SETTINGS),
    'eval': dict(EVAL_SETTINGS),
    'losses': {
        'weights': dict(LOSS_WEIGHTS),
    },
}

eval_config = {
    'seed': SEED,
    'device': device_name,
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': IMAGE_SIZE,
        'intrinsics': intrinsics,
        'pairing': dict(PAIRING),
        'splits': {
            'val': {'scenes': val_scenes},
        },
    },
    'model': {
        'depth': dict(DEPTH_MODEL_SETTINGS),
    },
    'eval': dict(EVAL_SETTINGS),
    'losses': {
        'weights': dict(LOSS_WEIGHTS),
    },
}

train_config_path = CONFIG_DIR / f'train_{EXPERIMENT_TAG}.yaml'
eval_config_path = CONFIG_DIR / f'eval_{EXPERIMENT_TAG}.yaml'
with train_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(train_config, handle, sort_keys=False)
with eval_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(eval_config, handle, sort_keys=False)

FAIR_EVAL_ARGS = [
    '--config', str(eval_config_path),
    '--max-batches', str(EVAL_SETTINGS['max_batches']),
]

# Default future-inference input patterns (edit as needed after training)
INFERENCE_INPUT_GLOBS = [
    str(TARGET_DATA_ROOT / scene_ids[0] / 'color' / '*.jpg')
]
INFERENCE_MAX_IMAGES = 16

print('device:', device_name)
print('train config:', train_config_path)
print('eval config:', eval_config_path)
print('train scenes:', len(train_scenes), 'val scenes:', len(val_scenes))
print('fair eval args:', FAIR_EVAL_ARGS)
print('depth mode:', DEPTH_MODE)
print('shared depth settings:', DEPTH_MODEL_SETTINGS)
print('pretrained depth checkpoint:', DEFAULT_DEPTH_CKPT)
print('loss weights:', LOSS_WEIGHTS)
print('checkpoint sweep enabled:', CHECKPOINT_SWEEP_ENABLED)
print('sweep primary metric:', SWEEP_PRIMARY_METRIC)
print('sweep coarse/full max_batches:', SWEEP_COARSE_MAX_BATCHES, SWEEP_FULL_MAX_BATCHES)


## Metric Parser

The evaluation CLI prints metrics to stdout. This helper turns that text into a dictionary for comparison and export.


In [14]:
def parse_metric_output(stdout_text: str):
    metrics = {}
    for line in stdout_text.splitlines():
        if ': ' not in line:
            continue
        key, value = line.split(': ', 1)
        try:
            metrics[key.strip()] = float(value.strip())
        except ValueError:
            continue
    return metrics


## Adapted-Only Evaluation Flow

Baseline evaluation is intentionally removed in this notebook.

The flow is:
1. train adapted model,
2. sweep checkpoints for convergence,
3. run final adapted evaluation using the selected best checkpoint.


In [ ]:
print('[status] baseline evaluation skipped by design (adapted-only workflow)')

In [ ]:
print('[status] no baseline metric artifact is produced in this adapted-only workflow')


## Train The Phase 1 Adaptation Model

This is the proposed improvement step: geometry-first round-trip supervision over a mostly frozen depth backbone with a lightweight adaptation head.


In [17]:
print('[status] starting Phase 1 training run')
train_run = run(
    ['python', '-m', 'eaglevision.cli.train', '--config', str(train_config_path)],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('train return code:', train_run.returncode)
if train_run.returncode != 0:
    raise RuntimeError('Training failed. Read streamed output above.')


[status] starting Phase 1 training run
$ python -m eaglevision.cli.train --config /kaggle/working/EagleVision/outputs/kaggle_configs/train_phase1_kaggle_heavy_v1.yaml
[status] cwd=/kaggle/working/EagleVision
[status] command started
Loading training config from /kaggle/working/EagleVision/outputs/kaggle_configs/train_phase1_kaggle_heavy_v1.yaml
Training on device=cuda
Built train dataset with 3454 pairs
Writing outputs to /kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1

epoch 1/70: 100%|██████████| 3454/3454 [04:04<00:00, 14.12it/s]

epoch 2/70: 100%|██████████| 3454/3454 [02:33<00:00, 22.56it/s]

epoch 3/70: 100%|██████████| 3454/3454 [02:36<00:00, 22.11it/s]

epoch 4/70: 100%|██████████| 3454/3454 [02:37<00:00, 21.92it/s]

epoch 5/70: 100%|██████████| 3454/3454 [02:35<00:00, 22.22it/s]

epoch 6/70: 100%|██████████| 3454/3454 [02:33<00:00, 22.52it/s]

epoch 7/70: 100%|██████████| 3454/3454 [02:33<00:00, 22.57it/s]

epoch 8/70: 100%|██████████| 3454/3454 [02:32<00:00, 22.61i

In [18]:
checkpoint_dir = TRAIN_RUN_OUTPUT_DIR / 'checkpoints'
checkpoints = sorted(checkpoint_dir.glob('*.pt'))
print('checkpoints:', checkpoints)

if not checkpoints:
    raise RuntimeError(f'No checkpoints found in {checkpoint_dir}')

latest_checkpoint = checkpoints[-1]
latest_checkpoint


checkpoints: [PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_001_final_step_0003454.pt'), PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_002_final_step_0006908.pt'), PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_003_final_step_0010362.pt'), PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_004_final_step_0013816.pt'), PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_005_final_step_0017270.pt'), PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_006_final_step_0020724.pt'), PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_007_final_step_0024178.pt'), PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_008_final_step_0027632.pt'), PosixPath('/kaggle/working/EagleVision/outputs/pha

PosixPath('/kaggle/working/EagleVision/outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_070_final_step_0241780.pt')

## Convergence Sweep And Final Adapted Evaluation

This stage evaluates multiple epoch checkpoints, ranks them by validation metrics, and picks a deterministic best checkpoint for final reporting.


In [ ]:
import re

print('[status] starting checkpoint sweep for convergence selection')

ckpt_dir = TRAIN_RUN_OUTPUT_DIR / 'checkpoints'
all_checkpoints = sorted(ckpt_dir.glob('epoch_*_final_step_*.pt'))
if not all_checkpoints:
    all_checkpoints = sorted(ckpt_dir.glob('*.pt'))

if not all_checkpoints:
    raise RuntimeError(f'No checkpoints found in {ckpt_dir}')

def _epoch_from_name(path: Path):
    match = re.search(r'epoch_(\d+)_', path.name)
    return int(match.group(1)) if match else None

def _run_adapted_eval(checkpoint_path: Path, max_batches: int):
    eval_run = run(
        [
            'python', '-m', 'eaglevision.cli.eval',
            '--config', str(eval_config_path),
            '--max-batches', str(int(max_batches)),
            '--checkpoint', str(checkpoint_path),
        ],
        cwd=REPO_DIR,
        capture=True,
        check=False,
    )
    metrics = parse_metric_output(eval_run.stdout)
    return eval_run, metrics

if CHECKPOINT_SWEEP_ENABLED:
    epoch_ckpts = [ckpt for ckpt in all_checkpoints if _epoch_from_name(ckpt) is not None]
    if not epoch_ckpts:
        epoch_ckpts = list(all_checkpoints)

    stride = max(1, int(SWEEP_EPOCH_STRIDE))
    coarse_candidates = [ckpt for ckpt in epoch_ckpts if (_epoch_from_name(ckpt) is None or _epoch_from_name(ckpt) % stride == 0)]

    # Ensure boundary checkpoints are included.
    coarse_candidates = [epoch_ckpts[0], *coarse_candidates, epoch_ckpts[-1]]

    # Deduplicate while preserving order.
    seen = set()
    coarse_checkpoints = []
    for ckpt in coarse_candidates:
        key = str(ckpt)
        if key in seen:
            continue
        seen.add(key)
        coarse_checkpoints.append(ckpt)
else:
    coarse_checkpoints = [all_checkpoints[-1]]

print('[status] coarse sweep checkpoints:', len(coarse_checkpoints))

sweep_rows = []
for index, ckpt in enumerate(coarse_checkpoints, start=1):
    print(f'[status] coarse eval {index}/{len(coarse_checkpoints)}: {ckpt.name}')
    eval_run, metrics = _run_adapted_eval(ckpt, SWEEP_COARSE_MAX_BATCHES)
    row = {
        'stage': 'coarse',
        'checkpoint': str(ckpt),
        'checkpoint_name': ckpt.name,
        'epoch': _epoch_from_name(ckpt),
        'max_batches': int(SWEEP_COARSE_MAX_BATCHES),
        'eval_return_code': int(eval_run.returncode),
        'status': 'ok' if eval_run.returncode == 0 else 'failed',
    }
    row.update(metrics)
    sweep_rows.append(row)

primary = SWEEP_PRIMARY_METRIC
coarse_ok = [r for r in sweep_rows if r.get('status') == 'ok' and primary in r and isinstance(r.get(primary), (int, float))]
if not coarse_ok:
    raise RuntimeError('No successful coarse checkpoint evaluations with the primary metric available.')

coarse_ranked = sorted(coarse_ok, key=lambda r: (r[primary], r.get('rmse', float('inf')), -(r.get('epoch') or -1)))
topk = max(1, int(SWEEP_TOPK_FOR_FULL_EVAL))
full_candidates = coarse_ranked[:topk]
print('[status] full-eval candidate checkpoints:', len(full_candidates))

for index, coarse_row in enumerate(full_candidates, start=1):
    ckpt = Path(coarse_row['checkpoint'])
    print(f'[status] full eval {index}/{len(full_candidates)}: {ckpt.name}')
    eval_run, metrics = _run_adapted_eval(ckpt, SWEEP_FULL_MAX_BATCHES)
    row = {
        'stage': 'full',
        'checkpoint': str(ckpt),
        'checkpoint_name': ckpt.name,
        'epoch': _epoch_from_name(ckpt),
        'max_batches': int(SWEEP_FULL_MAX_BATCHES),
        'eval_return_code': int(eval_run.returncode),
        'status': 'ok' if eval_run.returncode == 0 else 'failed',
    }
    row.update(metrics)
    sweep_rows.append(row)

full_ok = [r for r in sweep_rows if r.get('stage') == 'full' and r.get('status') == 'ok' and primary in r]
if not full_ok:
    print('[status] no successful full-stage runs; falling back to best coarse candidate')
    best_row = coarse_ranked[0]
else:
    best_row = sorted(
        full_ok,
        key=lambda r: (r[primary], r.get('rmse', float('inf')), -(r.get('epoch') or -1)),
    )[0]

best_checkpoint = Path(best_row['checkpoint'])
adapted_metrics = {
    key: value
    for key, value in best_row.items()
    if key not in {'stage', 'checkpoint', 'checkpoint_name', 'epoch', 'max_batches', 'eval_return_code', 'status'}
    and isinstance(value, (int, float))
}

print('[status] selected best checkpoint:', best_checkpoint)
print('[status] primary metric', primary, '=>', best_row.get(primary))


In [ ]:
import pandas as pd

print('[status] saving checkpoint sweep artifacts')

checkpoint_sweep_csv_path = TRAIN_RUN_OUTPUT_DIR / 'checkpoint_sweep.csv'
checkpoint_sweep_json_path = TRAIN_RUN_OUTPUT_DIR / 'checkpoint_sweep.json'
best_checkpoint_path = TRAIN_RUN_OUTPUT_DIR / 'best_checkpoint.yaml'
adapted_metrics_path = TRAIN_RUN_OUTPUT_DIR / 'adapted_metrics.yaml'

sweep_df = pd.DataFrame(sweep_rows)
if not sweep_df.empty:
    sweep_df = sweep_df.sort_values(['stage', 'epoch', 'checkpoint_name'], na_position='last').reset_index(drop=True)
sweep_df.to_csv(checkpoint_sweep_csv_path, index=False)

with checkpoint_sweep_json_path.open('w', encoding='utf-8') as handle:
    json.dump(sweep_rows, handle, indent=2)

best_checkpoint_payload = {
    'checkpoint': str(best_checkpoint),
    'primary_metric': SWEEP_PRIMARY_METRIC,
    'selected_row': best_row,
    'sweep_config': {
        'enabled': CHECKPOINT_SWEEP_ENABLED,
        'coarse_max_batches': SWEEP_COARSE_MAX_BATCHES,
        'full_max_batches': SWEEP_FULL_MAX_BATCHES,
        'epoch_stride': SWEEP_EPOCH_STRIDE,
        'topk_for_full_eval': SWEEP_TOPK_FOR_FULL_EVAL,
    },
}

with best_checkpoint_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(best_checkpoint_payload, handle, sort_keys=False)

with adapted_metrics_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(adapted_metrics, handle, sort_keys=False)

print('selected checkpoint:', best_checkpoint)
print('saved:', checkpoint_sweep_csv_path)
print('saved:', checkpoint_sweep_json_path)
print('saved:', best_checkpoint_path)
print('saved:', adapted_metrics_path)

display(sweep_df.head(30))
adapted_metrics


## Adapted Metrics Summary

This section summarizes the selected adapted-checkpoint metrics and saves compact analysis artifacts for tracking across runs.


In [ ]:
import numpy as np
import pandas as pd

metric_rows = [
    {'metric': metric_name, 'value': float(metric_value)}
    for metric_name, metric_value in sorted(adapted_metrics.items())
]
df = pd.DataFrame(metric_rows)

lower_better = {
    'target_rgb_l1', 'cycle_rgb_l1', 'reprojection_depth',
    'depth_l1', 'rmse', 'abs_rel',
    'loss_total', 'loss_target_rgb', 'loss_cycle_rgb', 'loss_cycle_depth', 'loss_target_depth',
}
higher_better = {'psnr', 'ssim'}

def direction(metric: str) -> str:
    if metric in lower_better:
        return 'lower_better'
    if metric in higher_better:
        return 'higher_better'
    return 'unknown'

df['direction'] = df['metric'].map(direction)
df = df.sort_values('metric').reset_index(drop=True)

print('selected checkpoint:', best_checkpoint)
print('num parsed metrics:', len(df))
display(df)


In [ ]:
adapted_summary_json_path = TRAIN_RUN_OUTPUT_DIR / 'adapted_metrics_summary.json'
adapted_summary_csv_path = TRAIN_RUN_OUTPUT_DIR / 'adapted_metrics_summary.csv'

summary_payload = {
    'experiment_tag': EXPERIMENT_TAG,
    'best_checkpoint': str(best_checkpoint),
    'metrics': adapted_metrics,
}

with adapted_summary_json_path.open('w', encoding='utf-8') as handle:
    json.dump(summary_payload, handle, indent=2)

df.to_csv(adapted_summary_csv_path, index=False)

print(adapted_summary_json_path)
print(adapted_summary_csv_path)


## Result Analysis And Consolidation Plots

These cells summarize adapted metrics from the selected checkpoint and consolidate adapted-only metrics across prior runs.


In [ ]:
import matplotlib.pyplot as plt

tracked = df[df['direction'] != 'unknown'].copy()
print(f'tracked metrics with known optimization direction: {len(tracked)}')

if not tracked.empty:
    direction_counts = tracked['direction'].value_counts().to_dict()
    print('direction counts:', direction_counts)

tracked_display = tracked[['metric', 'value', 'direction']].sort_values('metric').reset_index(drop=True)
display(tracked_display)


In [ ]:
plot_df = df.copy().sort_values('value', ascending=True).reset_index(drop=True)

color_map = {
    'lower_better': '#1f77b4',
    'higher_better': '#ff7f0e',
    'unknown': '#7f7f7f',
}
colors = [color_map.get(d, '#7f7f7f') for d in plot_df['direction']]

plt.figure(figsize=(10, max(6, 0.35 * len(plot_df))))
plt.barh(plot_df['metric'], plot_df['value'], color=colors)
plt.title('Selected Adapted Checkpoint Metrics')
plt.xlabel('Metric Value')
plt.tight_layout()
plt.show()


In [ ]:
focus_metrics = [
    'target_rgb_l1', 'cycle_rgb_l1', 'reprojection_depth',
    'depth_l1', 'rmse', 'abs_rel', 'psnr', 'ssim',
]
focus = df[df['metric'].isin(focus_metrics)].copy()
focus = focus.set_index('metric').loc[[m for m in focus_metrics if m in set(focus['metric'])]].reset_index()

plt.figure(figsize=(12, 4.8))
plt.bar(focus['metric'], focus['value'], color='#2ca02c')
plt.xticks(rotation=35, ha='right')
plt.title('Selected Adapted Checkpoint (Focus Metrics)')
plt.tight_layout()
plt.show()

print('Direction reference: lower is better for L1/RMSE/AbsRel/reprojection; higher is better for PSNR/SSIM.')


In [ ]:
# Optional: aggregate adapted metrics across runs from outputs/**/adapted_metrics.yaml
all_metric_files = sorted((REPO_DIR / 'outputs').rglob('adapted_metrics.yaml'))
print('found adapted metric files:', len(all_metric_files))
for pth in all_metric_files[:10]:
    print('-', pth)

rows = []
for metrics_path in all_metric_files:
    run_name = metrics_path.parent.name
    with metrics_path.open('r', encoding='utf-8') as f:
        data = yaml.safe_load(f) or {}
    for metric_name, metric_value in data.items():
        try:
            rows.append({'run': run_name, 'metric': metric_name, 'value': float(metric_value)})
        except (TypeError, ValueError):
            continue

if rows:
    runs_df = pd.DataFrame(rows)
    tracked = runs_df[runs_df['metric'].isin(focus_metrics)].copy()
    if not tracked.empty:
        summary = tracked.groupby('metric')['value'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
        display(summary.sort_values('metric'))

        plt.figure(figsize=(11, 4))
        ordered = summary.sort_values('mean')
        plt.bar(ordered['metric'], ordered['mean'], color='#1f77b4')
        plt.xticks(rotation=35, ha='right')
        plt.title('Mean Adapted Metric Value Across Runs')
        plt.tight_layout()
        plt.show()
    else:
        print('No focus metrics found in aggregated adapted metric files.')
else:
    print('No adapted_metrics.yaml files found yet. Re-run this cell after multiple experiments.')


## Adapted-Only Depth Visualization (Input + Novel View)

This section shows adapted-model depth estimation visuals only:
- source input RGB + adapted source depth
- forward-warped novel RGB + adapted novel depth

Note: forward warp is sparse geometry, so dark regions in novel RGB are expected holes (no inpainting in Phase 1).
In relative mode, absolute numeric depth is arbitrary; compare spatial structure rather than absolute scale.


In [27]:
import os, sys
from pathlib import Path
import importlib

REPO_DIR = Path("/kaggle/working/EagleVision")
SRC_DIR = REPO_DIR / "src"

# Make both eaglevision (src) and baseline (repo root) importable
for p in (str(REPO_DIR), str(SRC_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTHONPATH"] = f"{REPO_DIR}:{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"
importlib.invalidate_caches()

print("REPO in path:", str(REPO_DIR) in sys.path)
print("SRC in path :", str(SRC_DIR) in sys.path)


REPO in path: True
SRC in path : True


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from eaglevision.data.pair_sampler import PairSamplingConfig
from eaglevision.data.scannet_dataset import ScanNetPairDataset
from eaglevision.engine.checkpointing import load_checkpoint
from eaglevision.models.depth.depth_anything_wrapper import DepthAnythingWithAdapter
from eaglevision.models.nvs.geometric_warp import GeometricWarper
from eaglevision.models.rt_depthnvs import RoundTripDepthNVS

viz_device = torch.device(device_name if torch.cuda.is_available() else 'cpu')

if 'best_checkpoint' not in globals() or best_checkpoint is None:
    if 'latest_checkpoint' in globals() and latest_checkpoint is not None:
        best_checkpoint = latest_checkpoint
    else:
        checkpoint_candidates = sorted((TRAIN_RUN_OUTPUT_DIR / 'checkpoints').glob('*.pt'))
        if not checkpoint_candidates:
            raise RuntimeError('No training checkpoint available for adapted-depth visualization.')
        best_checkpoint = checkpoint_candidates[-1]

pair_cfg = PairSamplingConfig(**PAIRING)
base_intrinsics = np.array(intrinsics, dtype=np.float32)

viz_scenes = list(val_scenes)
if not viz_scenes:
    viz_scenes = list(train_scenes[:1])

viz_dataset = ScanNetPairDataset(
    root=TARGET_DATA_ROOT,
    scenes=viz_scenes,
    image_size=tuple(IMAGE_SIZE),
    intrinsics=base_intrinsics,
    pair_config=pair_cfg,
)
if len(viz_dataset) == 0:
    viz_dataset = ScanNetPairDataset(
        root=TARGET_DATA_ROOT,
        scenes=list(train_scenes),
        image_size=tuple(IMAGE_SIZE),
        intrinsics=base_intrinsics,
        pair_config=pair_cfg,
    )
if len(viz_dataset) == 0:
    raise RuntimeError('No valid pairs available for visualization. Relax pairing controls in the single control cell.')

sample = viz_dataset[0]

source_rgb = sample['source_rgb'].unsqueeze(0).to(viz_device)
source_depth = sample['source_depth'].unsqueeze(0).to(viz_device)
source_k = sample['source_intrinsics'].unsqueeze(0).to(viz_device)
target_k = sample['target_intrinsics'].unsqueeze(0).to(viz_device)
source_pose = sample['source_pose'].unsqueeze(0).to(viz_device)
target_pose = sample['target_pose'].unsqueeze(0).to(viz_device)

t_s2t = target_pose @ torch.inverse(source_pose)
warper = GeometricWarper().to(viz_device).eval()

adapted_depth_model = DepthAnythingWithAdapter(**DEPTH_MODEL_SETTINGS).to(viz_device)
adapted_roundtrip = RoundTripDepthNVS(adapted_depth_model).to(viz_device).eval()
load_checkpoint(best_checkpoint, adapted_roundtrip)

with torch.no_grad():
    forward = warper(source_rgb, source_depth, source_k, target_k, t_s2t)
    novel_rgb = forward['warped_rgb']

    adapted_source_pack = adapted_roundtrip.depth_model(source_rgb)
    adapted_novel_pack = adapted_roundtrip.depth_model(novel_rgb)

    source_depth_adapted = adapted_source_pack['adapted_depth'][0].detach().cpu().numpy()
    novel_depth_adapted = adapted_novel_pack['adapted_depth'][0].detach().cpu().numpy()

source_rgb_np = source_rgb[0].detach().cpu().permute(1, 2, 0).numpy()
novel_rgb_np = novel_rgb[0].detach().cpu().permute(1, 2, 0).numpy()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rgb_for_display(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, 0.0, 1.0)
    lo, hi = np.percentile(x, [1, 99])
    x = np.clip((x - lo) / (hi - lo + 1e-6), 0.0, 1.0)
    return x ** (1 / 2.2)

def _depth_valid(a: np.ndarray) -> np.ndarray:
    return np.isfinite(a) & (a > 0)

def _depth_range(a: np.ndarray):
    v = a[_depth_valid(a)]
    if v.size == 0:
        return 0.0, 1.0
    return np.percentile(v, 2), np.percentile(v, 98)

src_vmin, src_vmax = _depth_range(source_depth_adapted)
nov_vmin, nov_vmax = _depth_range(novel_depth_adapted)

source_rgb_vis = rgb_for_display(source_rgb_np)
novel_rgb_vis = rgb_for_display(novel_rgb_np)

holes = np.all(novel_rgb_np <= 1e-6, axis=-1)
novel_rgb_vis = novel_rgb_vis.copy()
novel_rgb_vis[holes] = np.array([0.15, 0.15, 0.15], dtype=np.float32)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

axes[0, 0].imshow(source_rgb_vis)
axes[0, 0].set_title('Source Input RGB')
axes[0, 0].axis('off')

im_src = axes[0, 1].imshow(source_depth_adapted, cmap='magma', vmin=src_vmin, vmax=src_vmax)
axes[0, 1].set_title('Source Depth: Adapted')
axes[0, 1].axis('off')

axes[1, 0].imshow(novel_rgb_vis)
axes[1, 0].set_title('Novel View RGB (Forward Warp)')
axes[1, 0].axis('off')

im_nov = axes[1, 1].imshow(novel_depth_adapted, cmap='magma', vmin=nov_vmin, vmax=nov_vmax)
axes[1, 1].set_title('Novel View Depth: Adapted')
axes[1, 1].axis('off')

fig.colorbar(im_src, ax=[axes[0, 1]], fraction=0.046, pad=0.04, label='Depth (Source scale)')
fig.colorbar(im_nov, ax=[axes[1, 1]], fraction=0.046, pad=0.04, label='Depth (Novel scale)')

adapted_depth_panel = TRAIN_RUN_OUTPUT_DIR / 'adapted_input_novel_depth_panel.png'
fig.savefig(adapted_depth_panel, dpi=150, bbox_inches='tight')
plt.show()
print('saved:', adapted_depth_panel)


In [ ]:
for name, d in [
    ('source_depth_adapted', source_depth_adapted),
    ('novel_depth_adapted', novel_depth_adapted),
]:
    v = d[np.isfinite(d) & (d > 0)]
    if v.size == 0:
        print(name, 'no valid values')
        continue
    print(name, 'min', float(v.min()), 'max', float(v.max()), 'std', float(v.std()))


## Optional Adapted-Checkpoint Inference Helper

This helper runs the **trained adapted model checkpoint** on arbitrary image patterns.

Edit `INFERENCE_INPUT_GLOBS` (in the control cell) to point to any dataset paths for future comparisons.
Outputs are written to `outputs/<run_tag>/inference/` and indexed in `inference_manifest.csv`.


In [ ]:
from glob import glob

import cv2
import pandas as pd
import torch

from eaglevision.engine.checkpointing import load_checkpoint
from eaglevision.models.depth.depth_anything_wrapper import DepthAnythingWithAdapter
from eaglevision.models.rt_depthnvs import RoundTripDepthNVS
from eaglevision.utils.visualization import depth_to_colormap

infer_device = torch.device(device_name if torch.cuda.is_available() else 'cpu')

if 'best_checkpoint' not in globals() or best_checkpoint is None:
    if 'latest_checkpoint' in globals() and latest_checkpoint is not None:
        best_checkpoint = latest_checkpoint
    else:
        checkpoint_candidates = sorted((TRAIN_RUN_OUTPUT_DIR / 'checkpoints').glob('*.pt'))
        if not checkpoint_candidates:
            raise RuntimeError('No checkpoint available for adapted inference helper.')
        best_checkpoint = checkpoint_candidates[-1]

input_patterns = list(INFERENCE_INPUT_GLOBS) if 'INFERENCE_INPUT_GLOBS' in globals() else []
if not input_patterns:
    input_patterns = [str(TARGET_DATA_ROOT / scene_ids[0] / 'color' / '*.jpg')]

image_paths = []
for pattern in input_patterns:
    image_paths.extend(sorted(glob(pattern)))

# Deduplicate and cap.
image_paths = list(dict.fromkeys(image_paths))
max_images = int(INFERENCE_MAX_IMAGES) if 'INFERENCE_MAX_IMAGES' in globals() else 16
if max_images > 0:
    image_paths = image_paths[:max_images]

if not image_paths:
    raise RuntimeError('No images found for adapted inference helper. Update INFERENCE_INPUT_GLOBS.')

inference_dir = TRAIN_RUN_OUTPUT_DIR / 'inference'
inference_dir.mkdir(parents=True, exist_ok=True)

adapted_depth_model = DepthAnythingWithAdapter(**DEPTH_MODEL_SETTINGS).to(infer_device)
adapted_roundtrip = RoundTripDepthNVS(adapted_depth_model).to(infer_device).eval()
load_checkpoint(best_checkpoint, adapted_roundtrip)

rows = []
for index, image_path in enumerate(image_paths, start=1):
    print(f'[infer] {index}/{len(image_paths)} {image_path}')
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        print('[infer] skipped unreadable image:', image_path)
        continue

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb).float().permute(2, 0, 1).unsqueeze(0).to(infer_device) / 255.0

    with torch.no_grad():
        depth = adapted_roundtrip.depth_model(tensor)['adapted_depth'][0]
    preview = depth_to_colormap(depth.detach().cpu())

    stem = Path(image_path).stem
    output_path = inference_dir / f'{stem}_adapted_depth.png'
    cv2.imwrite(str(output_path), cv2.cvtColor(preview, cv2.COLOR_RGB2BGR))

    rows.append(
        {
            'input_path': image_path,
            'output_path': str(output_path),
            'checkpoint': str(best_checkpoint),
            'experiment_tag': EXPERIMENT_TAG,
            'height': int(rgb.shape[0]),
            'width': int(rgb.shape[1]),
        }
    )

if not rows:
    raise RuntimeError('Inference helper ran but no outputs were produced.')

manifest_path = inference_dir / 'inference_manifest.csv'
pd.DataFrame(rows).to_csv(manifest_path, index=False)
print('inference outputs:', len(rows))
print('manifest:', manifest_path)
manifest_path


## Produced Artifacts

List the main artifacts generated by this run.


In [32]:
artifact_paths = sorted(str(path.relative_to(REPO_DIR)) for path in TRAIN_RUN_OUTPUT_DIR.rglob('*'))
artifact_paths[:200]


['outputs/phase1_kaggle_heavy_v1/adapted_metrics.yaml',
 'outputs/phase1_kaggle_heavy_v1/baseline_metrics.yaml',
 'outputs/phase1_kaggle_heavy_v1/baseline_vs_adapted_input_novel_depth_panel.png',
 'outputs/phase1_kaggle_heavy_v1/checkpoints',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_001_final_step_0003454.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_002_final_step_0006908.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_003_final_step_0010362.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_004_final_step_0013816.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_005_final_step_0017270.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_006_final_step_0020724.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_007_final_step_0024178.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_008_final_step_0027632.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/epoch_009_final_step_0031086.pt',
 'outputs/phase1_kaggle_heavy_v1/checkpoints/e

## Package Outputs For Export

This creates a zip file containing the run artifacts, metric summaries, and checkpoint outputs for later reuse.


In [33]:
archive_base = REPO_DIR / 'outputs' / 'phase1_kaggle_complete_export'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=TRAIN_RUN_OUTPUT_DIR)
archive_path


'/kaggle/working/EagleVision/outputs/phase1_kaggle_complete_export.zip'

## Final Summary

This notebook demonstrates an adapted-only long-run Phase 1 workflow:

- ablation-best hyperparameter configuration
- long training run
- convergence-oriented checkpoint sweep and deterministic checkpoint selection
- final adapted evaluation
- adapted-only metric summaries
- adapted-checkpoint inference helper for future dataset/quality comparisons

If the selected adapted checkpoint improves geometry-facing metrics while maintaining acceptable depth quality, that supports the central project hypothesis and provides a reusable model for downstream comparison studies.
